In [ ]:
import os
import random
import numpy as np
import cv2
from sklearn.decomposition import PCA
import tensorflow as tf
from tensorflow.keras import layers, Input, Model
from sklearn.model_selection import train_test_split

In [ ]:
# Data Loading with Class Balance
# -----------------------------
def load_and_sample_images_weighted(base_path, total_samples):
    images = []
    labels = []
    class_names = []
    class_distributions = {}

    for label, folder in enumerate(sorted(os.listdir(base_path))):
        class_names.append(folder)
        folder_path = os.path.join(base_path, folder)
        class_images = []
        for file in os.listdir(folder_path):
            img_path = os.path.join(folder_path, file)
            img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
            img = cv2.resize(img, (128, 128))
            class_images.append(img.flatten())
        class_distributions[label] = class_images

    num_classes = len(class_distributions)
    samples_per_class = total_samples // num_classes

    for label, imgs in class_distributions.items():
        sample_size = min(samples_per_class, len(imgs))
        sampled_images = random.sample(imgs, sample_size)
        images.extend(sampled_images)
        labels.extend([label] * sample_size)

    return np.array(images), np.array(labels), class_names


In [ ]:
# PINN Model
# -----------------------------
class PINN(tf.keras.Model):
    def __init__(self):
        super(PINN, self).__init__()
        self.dense1 = tf.keras.layers.Dense(64, activation='tanh')
        self.dense2 = tf.keras.layers.Dense(64, activation='tanh')
        self.dense3 = tf.keras.layers.Dense(2)
        self.c1 = tf.Variable(2.0, trainable=True, dtype=tf.float32)
        self.c3 = tf.Variable(-0.2, trainable=True, dtype=tf.float32)

    def call(self, inputs):
        x = self.dense1(inputs)
        x = self.dense2(x)
        return self.dense3(x)

# -----------------------------
# Differentiable Laplacian in TensorFlow
# -----------------------------
def laplacian_tf(u, h=1.0):
    u_xx = (tf.roll(u, shift=-1, axis=1) - 2*u + tf.roll(u, shift=1, axis=1)) / (h**2)
    u_yy = (tf.roll(u, shift=-1, axis=0) - 2*u + tf.roll(u, shift=1, axis=0)) / (h**2)
    return u_xx + u_yy


In [ ]:
# CLG PDE Loss for PINN
# -----------------------------
def pinn_loss(model, image):
    h, w = image.shape
    grid_x, grid_y = np.meshgrid(np.linspace(0, 1, w), np.linspace(0, 1, h))
    coords = np.stack([grid_x.flatten(), grid_y.flatten()], axis=-1).astype(np.float32)
    inputs = tf.convert_to_tensor(coords)

    with tf.GradientTape(persistent=True) as tape:
        tape.watch(model.c1)
        tape.watch(model.c3)

        outputs = model(inputs)
        v1 = tf.reshape(outputs[:, 0], (h, w))
        v2 = tf.reshape(outputs[:, 1], (h, w))

        v1_lap = laplacian_tf(v1)
        v2_lap = laplacian_tf(v2)

        v_sq = v1**2 + v2**2

        # Full CLG residuals
        res1 = v1_lap + model.c1 * v1 - (v1 - model.c3 * v2) * v_sq
        res2 = v2_lap + model.c1 * v2 - (model.c3 * v1 + v2) * v_sq

        loss = tf.reduce_mean(res1**2 + res2**2)

    return loss


In [ ]:
# Train PINN for one ECG image
# -----------------------------
def learn_equation_parameters_pinn(ecg_image, epochs=50):
    img = ecg_image / 255.0
    model = PINN()
    optimizer = tf.keras.optimizers.Adam(learning_rate=0.01)

    for epoch in range(epochs):
        with tf.GradientTape() as tape:
            tape.watch(model.c1)
            tape.watch(model.c3)
            loss = pinn_loss(model, img)
        grads = tape.gradient(loss, [model.c1, model.c3] + model.trainable_variables)
        optimizer.apply_gradients(zip(grads, [model.c1, model.c3] + model.trainable_variables))

    return float(model.c1.numpy()), float(model.c3.numpy())


In [ ]:
# Paths
# -----------------------------
train_path = r"F:\fatemeh\phd\lesson\Linear Algebra\project\archive (4)\ECG_Image_data\train"
test_path = r"F:\fatemeh\phd\lesson\Linear Algebra\project\archive (4)\ECG_Image_data\test"
total_train_samples = 6000
total_test_samples = 1200


In [ ]:
# Step 1: Load Data
# -----------------------------
X_train, y_train, class_names = load_and_sample_images_weighted(train_path, total_train_samples)
X_test, y_test, _ = load_and_sample_images_weighted(test_path, total_test_samples)

X_train_imgs = X_train.reshape(-1, 128, 128)
X_test_imgs = X_test.reshape(-1, 128, 128)


In [ ]:
# Step 2: Extract PINN Features
# -----------------------------
print("[3] Extracting PINN parameters for training set...")
X_train_params = np.array([learn_equation_parameters_pinn(img) for img in X_train_imgs])

print("[3] Extracting PINN parameters for test set...")
X_test_params = np.array([learn_equation_parameters_pinn(img) for img in X_test_imgs])


In [ ]:
# -----------------------------
# c1 و c3
# -----------------------------
def print_pinn_stats(params, set_name):
    c1_values = params[:, 0]
    c3_values = params[:, 1]
    print(f"\n[{set_name} PINN Parameters Statistics]")
    print(f"c1 -> Mean: {np.mean(c1_values):.4f}, Std: {np.std(c1_values):.4f}, Min: {np.min(c1_values):.4f}, Max: {np.max(c1_values):.4f}")
    print(f"c3 -> Mean: {np.mean(c3_values):.4f}, Std: {np.std(c3_values):.4f}, Min: {np.min(c3_values):.4f}, Max: {np.max(c3_values):.4f}")

print_pinn_stats(X_train_params, "Train")
print_pinn_stats(X_test_params, "Test")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans

def plot_c1_c3_advanced(X_train_params, y_train, X_test_params, y_test, class_names,
                        n_clusters=3, s=22, alpha=0.55):
    """
    Advanced c1–c3 visualization:
      - per-class scatter
      - global density contours
      - KMeans cluster centers with labels
      - side-by-side Train vs Test comparison
    """
    sns.set(style="whitegrid")
    fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharex=True, sharey=True)

    sets = [
        ("Train", X_train_params, y_train, axes[0]),
        ("Test",  X_test_params,  y_test,  axes[1]),
    ]

    for title, Xp, yp, ax in sets:
        c1, c3 = Xp[:,0], Xp[:,1]

        
        sns.kdeplot(
            x=c1, y=c3, ax=ax,
            levels=6, linewidths=1.2, color="gray", fill=False, alpha=0.9
        )

        
        palette = sns.color_palette("tab10", len(class_names))
        for i, name in enumerate(class_names):
            mask = (yp == i)
            ax.scatter(c1[mask], c3[mask], s=s, alpha=alpha, label=name,
                       color=palette[i], edgecolors="none")

        
        km = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
        km.fit(Xp)
        centers = km.cluster_centers_  # shape: (k, 2)

        ax.scatter(centers[:,0], centers[:,1],
                   s=180, marker="X", color="black", label="Cluster Center")
        
        for j, (cx, cz) in enumerate(centers):
            ax.annotate(f"C{j+1}\n({cx:.2f}, {cz:.2f})",
                        (cx, cz), xytext=(6, 6), textcoords="offset points",
                        fontsize=9, weight="bold")

        ax.set_title(f"c1 vs c3 ({title} Set)", fontsize=13)
        ax.set_xlabel("c1")
        ax.set_ylabel("c3")
        ax.grid(True, alpha=0.25)

    
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="lower center", ncol=min(6, len(labels)), frameon=False)
    plt.tight_layout(rect=[0, 0.06, 1, 1])
    plt.show()


plot_c1_c3_advanced(X_train_params, y_train, X_test_params, y_test, class_names,
                    n_clusters=3, s=22, alpha=0.55)


In [ ]:
# Classifier with ONLY PINN features (c1, c3)
# =============================
from tensorflow.keras import layers, Input, Model
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, roc_curve, auc
from sklearn.preprocessing import label_binarize
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

# 1) Scale features 
scaler = StandardScaler()
X_train_pinn = scaler.fit_transform(X_train_params)   
X_test_pinn  = scaler.transform(X_test_params)        

# 2) Build a compact MLP
inp = Input(shape=(X_train_pinn.shape[1],)) 
x = layers.Dense(32, activation='relu')(inp)
x = layers.Dropout(0.2)(x)
x = layers.Dense(16, activation='relu')(x)
out = layers.Dense(len(class_names), activation='softmax')(x)

clf = Model(inputs=inp, outputs=out)
clf.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
print(clf.summary())

# 3) Train with early stopping
es = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
history = clf.fit(
    X_train_pinn, y_train,
    epochs=80, batch_size=64,
    validation_data=(X_test_pinn, y_test),
    callbacks=[es],
    verbose=1
)


In [ ]:
# 4) Evaluation – accuracy, report, confusion matrix
y_proba = clf.predict(X_test_pinn)
y_pred  = np.argmax(y_proba, axis=1)

acc = accuracy_score(y_test, y_pred)
print(f"\n[Test Accuracy] {acc:.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=class_names))

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(7,5))
sns.heatmap(cm, annot=True, fmt='d',
            xticklabels=class_names, yticklabels=class_names, cmap='Blues')
plt.title("Confusion Matrix – PINN-only")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.tight_layout()
plt.show()

In [ ]:
# 5) ROC (One-vs-Rest)
y_test_bin = label_binarize(y_test, classes=np.arange(len(class_names)))
fpr, tpr, roc_auc = {}, {}, {}
for i in range(y_test_bin.shape[1]):
    fpr[i], tpr[i], _ = roc_curve(y_test_bin[:, i], y_proba[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])

plt.figure(figsize=(8,6))
for i in range(len(class_names)):
    plt.plot(fpr[i], tpr[i], label=f"{class_names[i]} (AUC={roc_auc[i]:.2f})")
plt.plot([0,1],[0,1],'k--')
plt.title("ROC (PINN-only features)")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend(loc="lower right")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:

plt.figure(figsize=(8,4))
plt.plot(history.history['accuracy'], label='train acc')
plt.plot(history.history['val_accuracy'], label='val acc')
plt.title('Training History (PINN-only)')
plt.xlabel('Epoch'); plt.ylabel('Accuracy'); plt.legend(); plt.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()